# Credit Risk & Loan Default Prediction Walkthrough

This notebook demonstrates an end-to-end credit risk modeling workflow:
1. **Synthetic Data Generation**: Creating realistic borrower profiles with latent noise and calibrated default rate.
2. **Exploratory Data Analysis (EDA)**: Empirical default rates by loan purpose and credit score tier.
3. **Feature Engineering**: Amortisation monthly payments, loan-to-income, and payment-to-income ratios.
4. **Time-Based Splitting & Model Training**: Training a Logistic Regression scorecard baseline and an Isotonically Calibrated HistGradientBoosting model.
5. **Evaluation**: ROC-AUC, PR-AUC, Brier score, Kolmogorov-Smirnov (KS) statistic, and Gini coefficient.
6. **Probability Calibration**: Reliability diagrams comparing baseline vs. calibrated predictions.
7. **Approval Policy Optimization**: Establishing an 80% approval threshold on validation and evaluating on test.
8. **Decisioning & Explainability**: Scoring applicant profiles and generating adverse risk factors.

In [ ]:
import sys
from pathlib import Path

# Ensure project root and src directory are on sys.path
sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    brier_score_loss,
    roc_auc_score,
    roc_curve,
)
from sklearn.pipeline import Pipeline

from loan_default.data import generate
from loan_default.features import (
    ALL_FEATURES,
    CATEGORICAL,
    NUMERIC,
    add_derived,
    build_preprocessor,
)
from loan_default.model import predict

## 1. Data Generation

Generate 30,000 synthetic borrower profiles with risk-based pricing and latent unobserved noise.

In [ ]:
raw_df = generate(n=30000, seed=11)
print(f"Generated {len(raw_df):,} rows.")
print(f"Overall default rate: {raw_df['default'].mean():.2%}")
raw_df.head()

## 2. Exploratory Data Analysis (EDA)

Inspect the default rate across loan purposes and credit score bands.

In [ ]:
# Default rate by purpose
purpose_stats = (
    raw_df.groupby("purpose")["default"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "default_rate", "count": "total_loans"})
    .sort_values("default_rate", ascending=False)
)
print("Default Rate by Purpose:")
print(purpose_stats)

# Default rate by credit score band
score_bins = [300, 580, 670, 740, 800, 850]
score_labels = ["Poor (300-579)", "Fair (580-669)", "Good (670-739)", "Very Good (740-799)", "Exceptional (800-850)"]
raw_df["score_tier"] = pd.cut(raw_df["credit_score"], bins=score_bins, labels=score_labels, right=True)

tier_stats = (
    raw_df.groupby("score_tier", observed=False)["default"]
    .agg(["count", "mean"])
    .rename(columns={"mean": "default_rate", "count": "total_loans"})
)
print("\nDefault Rate by Credit Score Tier:")
print(tier_stats)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Purpose bar plot
axes[0].barh(purpose_stats.index, purpose_stats["default_rate"] * 100, color="#3b6e8c")
axes[0].set_xlabel("Default Rate (%)")
axes[0].set_title("Default Rate by Loan Purpose")
axes[0].grid(axis="x", alpha=0.3)

# Score tier bar plot
axes[1].bar(tier_stats.index.astype(str), tier_stats["default_rate"] * 100, color="#d95f02")
axes[1].set_ylabel("Default Rate (%)")
axes[1].set_title("Default Rate by Credit Score Tier")
axes[1].tick_params(axis="x", rotation=25)
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Feature Engineering

Compute amortisation monthly payment, loan-to-income, and payment-to-income ratios.

In [ ]:
df = add_derived(raw_df)
df[["loan_amount", "annual_income", "loan_to_income", "monthly_payment", "payment_to_income"]].head()

## 4. Time-Based Splitting & Model Training

Split chronologically by application date:
- **Train**: Months 1–18 (before 2025-07-01)
- **Validation**: Months 19–21 (2025-07-01 to 2025-10-01)
- **Test**: Months 22–24 (from 2025-10-01 onward)

Fit two architectures:
1. **Logistic Regression** (Scorecard baseline with balanced class weights)
2. **HistGradientBoostingClassifier** (Calibrated with 3-fold isotonic regression)

In [ ]:
train_df = df[df["application_date"] < "2025-07-01"].copy()
val_df = df[(df["application_date"] >= "2025-07-01") & (df["application_date"] < "2025-10-01")].copy()
test_df = df[df["application_date"] >= "2025-10-01"].copy()

print(f"Train records: {len(train_df):,} | Val records: {len(val_df):,} | Test records: {len(test_df):,}")

X_train, y_train = train_df[ALL_FEATURES], train_df["default"].to_numpy(dtype=int)
X_val, y_val = val_df[ALL_FEATURES], val_df["default"].to_numpy(dtype=int)
X_test, y_test = test_df[ALL_FEATURES], test_df["default"].to_numpy(dtype=int)

# 1. Scorecard Baseline
scorecard_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=42)),
])
scorecard_pipe.fit(X_train, y_train)

# 2. HistGradientBoosting Pipeline with 3-Fold Isotonic Calibration
hgb_pipe = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("classifier", HistGradientBoostingClassifier(random_state=42)),
])
hgb_calibrated = CalibratedClassifierCV(estimator=hgb_pipe, method="isotonic", cv=3)
hgb_calibrated.fit(X_train, y_train)

## 5. Metrics Evaluation

Evaluate discriminatory power (ROC-AUC, PR-AUC, KS statistic, Gini) and probabilistic accuracy (Brier score).

In [ ]:
def eval_predictions(y_true, y_prob):
    auc = roc_auc_score(y_true, y_prob)
    pr_auc = average_precision_score(y_true, y_prob)
    brier = brier_score_loss(y_true, y_prob)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    ks = np.max(tpr - fpr)
    gini = 2.0 * auc - 1.0
    return {"ROC-AUC": auc, "PR-AUC": pr_auc, "Brier": brier, "KS": ks, "Gini": gini}

scorecard_test_probs = scorecard_pipe.predict_proba(X_test)[:, 1]
hgb_test_probs = hgb_calibrated.predict_proba(X_test)[:, 1]

results = pd.DataFrame([
    {"Model": "Logistic Regression (Scorecard)", **eval_predictions(y_test, scorecard_test_probs)},
    {"Model": "HistGradientBoosting (Calibrated)", **eval_predictions(y_test, hgb_test_probs)},
]).set_index("Model")

results.round(4)

## 6. Calibration Plot (Reliability Diagram)

Verify that predicted probabilities reflect empirical default frequencies across probability buckets.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

for label, probs in [("Scorecard", scorecard_test_probs), ("HGB (Calibrated)", hgb_test_probs)]:
    fop, mpv = calibration_curve(y_test, probs, n_bins=10, strategy="uniform")
    ax.plot(mpv, fop, "s-", label=label, lw=2)

ax.plot([0, 1], [0, 1], "k--", label="Perfect Calibration", lw=1.5)
ax.set_xlabel("Mean Predicted Probability")
axes = ax.set_ylabel("Observed Fraction of Defaults")
ax.set_title("Probability Calibration (Reliability Diagram)")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Approval Policy Optimization

Determine a credit decision threshold on the validation set to achieve an 80% approval rate, then evaluate resulting default rates on holdout test data.

In [ ]:
val_probs = hgb_calibrated.predict_proba(X_val)[:, 1]
threshold_80 = float(np.percentile(val_probs, 80))

test_approved = hgb_test_probs <= threshold_80
test_approval_rate = np.mean(test_approved)
test_default_rate_approved = np.mean(y_test[test_approved])
test_default_rate_overall = np.mean(y_test)

policy_summary = pd.DataFrame([{
    "Cut-Off Threshold": threshold_80,
    "Test Approval Rate": f"{test_approval_rate:.1%}",
    "Default Rate (Approved Portfolio)": f"{test_default_rate_approved:.2%}",
    "Default Rate (Overall Benchmark)": f"{test_default_rate_overall:.2%}",
    "Risk Reduction Factor": f"{(test_default_rate_overall - test_default_rate_approved) / test_default_rate_overall:.1%}",
}])
policy_summary.T

## 8. Example Prediction & Decision Explanation

Score an individual loan applicant through the prediction pipeline, showing the decision, risk tier, and adverse action factors.

In [ ]:
bundle = {
    "models": {"hgb_calibrated": hgb_calibrated},
    "chosen_model_name": "hgb_calibrated",
    "threshold": threshold_80,
    "feature_lists": {"all": ALL_FEATURES, "numeric": NUMERIC, "categorical": CATEGORICAL},
    "version": "1.0.0",
}

sample_applicant = {
    "loan_amount": 25000.0,
    "annual_income": 55000.0,
    "age": 29,
    "employment_years": 3.0,
    "home_ownership": "rent",
    "purpose": "debt_consolidation",
    "term_months": 36,
    "interest_rate": 0.18,
    "debt_to_income": 0.48,
    "credit_score": 590.0,
    "num_open_accounts": 5,
    "num_delinquencies_2y": 1,
    "credit_utilization": 0.85,
}

decision_output = predict(bundle, sample_applicant)
print("Applicant Decision Summary:")
for key, val in decision_output.items():
    print(f"  {key}: {val}")